In [1]:
import pandas as pd


In [ ]:
import os
import pandas as pd
import numpy as np
from openai import OpenAI
from transformers import AutoTokenizer, AutoModel
import torch

# Initialize the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-distilroberta-v1', clean_up_tokenization_spaces=True)
model = AutoModel.from_pretrained('sentence-transformers/all-distilroberta-v1')

# Function to split text into chunks
def chunk_text(text, max_tokens=512):
    words = text.split()
    chunks = []
    start = 0
    
    while start < len(words):
        end = min(start + max_tokens, len(words))
        chunks.append(" ".join(words[start:end]))
        start = end
    
    return chunks

# Function to get query embedding
def get_query_embedding(question):
    print("Received question:", question)  # Debug: question input
    question_chunks = chunk_text(question, max_tokens=512)  

    question_embeddings = []
    '''
    for chunk in question_chunks:
        print("Processing chunk:", chunk[:100])  # Debug: chunk of text being processed
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True, padding=True, max_length=512)

        question_embedding = model(**inputs).last_hidden_state.mean(dim=1).detach().cpu().numpy().flatten()
        question_embeddings.append(question_embedding)
    '''

    #question_embedding = np.mean(question_embeddings, axis=0)
    inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=512)

    question_embedding = model(**inputs).last_hidden_state.mean(dim=1).detach().cpu().numpy().flatten()

    print("Question embedding shape:", question_embedding.shape)  # Debug: shape of question embedding

    return question_embedding

# Function to compute cosine similarity
def cosine_similarity(vec1, vec2):
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))

# Query and retrieval function
def retrieve_closest_text(question, embeddings_df, text_df, section="abstract"):
    question_embedding = get_query_embedding(question)
    
    # Calculate cosine similarity for each embedding in the specified section
    def calculate_similarity(x):
        if isinstance(x, str):
            try:
                page_embedding = np.array(eval(x), dtype=np.float32)
                return cosine_similarity(question_embedding, page_embedding)
            except Exception as e:
                print(f"Error evaluating embedding: {e}")
                return -np.inf
        else:
            return -np.inf  # Return -inf for invalid entries like NaN

    similarities = embeddings_df[section].apply(calculate_similarity)
    
    # Get indices of the top 5 highest similarities
    top_5_indices = similarities.nlargest(20).index
    top_5_texts = text_df.loc[top_5_indices, section]
    
    print(f"Top 5 matches in '{section}' section:")
    for i, text in enumerate(top_5_texts):
        print(f"Match {i+1}: {text[:500]}")  # Show the first 500 characters for readability
    
    # Return the closest match
    closest_idx = top_5_indices[0]
    closest_text = text_df.loc[closest_idx, section]
    
    return closest_text

# Read the CSV files
text_df = pd.read_csv('final_texts.csv')
embeddings_df = pd.read_csv('embeddings_prime.csv')

# Example usage
question1 = "When did Hurricane Maria hit Puerto Rico?"

closest_abstract = retrieve_closest_text(question1, embeddings_df, text_df, section="abstract")


In [ ]:
import os
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

max_seq_length = 512


# 2. Load Llama3 model
model, tokenizer = FastLanguageModel.from_pretrained(
    #model_name = "unsloth/llama-3-70b-bnb-4bit",
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
    cache_dir='/media/zman/extrahd/reu20024project/',
   
    device_map="auto"
)

model = FastLanguageModel.for_inference(model)


In [ ]:
# 3 Before training
def generate_text(text, model):
    inputs = tokenizer(text, return_tensors="pt").to("cuda:0")
    outputs = model.generate(**inputs, max_new_tokens=1028)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

In [ ]:
closest_abstract

In [ ]:
prompt = "Answer the question at the end with the provided context. Context: " + closest_abstract + ". " + question1
prompt

In [ ]:
response = generate_text(prompt, model)

In [ ]:
import textwrap

def print_wrapped_text(text, width=120):
    """
    Prints the given text with word wrapping to the specified width.

    Parameters:
    text (str): The text to be printed.
    width (int): The maximum width of each line (default is 120 characters).
    """
    wrapped_text = textwrap.fill(text, width=width)
    print(wrapped_text)

In [ ]:
print_wrapped_text(response)